In [3]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os 
from shapely.geometry import Point
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.patches import FancyArrowPatch
import matplotlib.image as mpimg


root = fr"C:\Users\eunic\Dropbox\sa_fires"
main_dir =  fr"{root}/proj_bureaucrats_farms"
int_path = fr"{main_dir}/data_output/intermediate"


In [5]:
# Import grid and states
grid = gpd.read_file( fr"{root}/proj_downwind/data_output/intermediate/1-grid-generation.shp" )
polygons = gpd.read_file( fr'{root}/data/input/ac_boundaries/Constituencies_Boundaries_Post_2008.shp' )


# Selecting States
sel_states = ["PUNJAB", "HARYANA",  "UTTAR PRADESH",  "BIHAR"]
sel_states_shp = polygons[ polygons.STATE_UT.isin( sel_states ) ]
fire_icon = mpimg.imread( f'{root}/proj_downwind/tex/paper/figures/fire.png')

# Acs selected
acs = gpd.read_file(fr'{int_path}/_0_2_3_ACs_right_shapefile.shp')
sel_polygon = acs.query('ac_uq_id == 780').copy()

In [ ]:
# Assuming sel_states_shp and polygons are already loaded
# Select the district with dist_id == 156
selected_acs = acs.query('ac_uq_id == 780').copy()

# Calculate the total bounds of sel_states_shp and add a margin
xmin, ymin, xmax, ymax = sel_states_shp.total_bounds
x_margin = 1.0  # 1 degree added to each side
y_margin = 1.0

# Set up the main plot
fig, ax = plt.subplots(figsize=(30, 30))

# Plot sel_states_shp with green fill and no edge
sel_states_shp.plot(ax=ax, color='#AFE1AF', edgecolor=None)

# Plot polygons with only boundaries in blue
polygons.boundary.plot(ax=ax, color='#336ece')

# Set the axis limits with added margins
ax.set_xlim(xmin - x_margin, xmax + x_margin)
ax.set_ylim(ymin - y_margin, ymax + y_margin)

# Add other plot customizations if needed
ax.axis('off')  # Turn off the axis

# Create an inset axis for the zoomed-in district
inset_ax = fig.add_axes([0.6, 0.45, 0.3, 0.3])  # [left, bottom, width, height] in figure coordinates

# Plot the selected district in the inset
polygons.plot(ax=inset_ax, color='#AFE1AF', edgecolor='black')

# Optionally, plot the surrounding polygons with boundaries in the inset (if needed)
polygons.boundary.plot(ax=inset_ax, color='#336ece', linewidth=0.5)

# adding the grid
grid.plot(ax=inset_ax, facecolor='none', edgecolor='black', linewidth=0.1)

# Set limits for the inset axis based on the bounds of the selected district with a margin
xmin_inset, ymin_inset, xmax_inset, ymax_inset = selected_acs.total_bounds
x_inset_margin = 0.1  # Adjust margin as needed for better visibility
y_inset_margin = 0.02

inset_ax.set_xlim(xmin_inset - x_inset_margin, xmax_inset + x_inset_margin)
inset_ax.set_ylim(ymin_inset - y_inset_margin, ymax_inset + y_inset_margin)

# Hide axis for the inset plot
inset_ax.axis('off')

# Add a red square around the inset plot
rect = Rectangle((0, 0), 1, 1, transform=inset_ax.transAxes,
                 color='red', fill=False, lw=5, zorder=9)
inset_ax.add_patch(rect)

# Calculate the centroid of the selected district
centroid = selected_acs.geometry.centroid.iloc[0]



# Add an arrow from the centroid to the midpoint of the inset's bottom line
arrow = FancyArrowPatch((centroid.x, centroid.y), (centroid.x + 5.4, centroid.y + 2.2),
                        transform=ax.transData,
                        color='red', arrowstyle='->', lw=3, zorder=10)
ax.add_patch(arrow)

# Save the code
fig.savefig( fr"{main_dir}/tex/paper/figures/map_grids.pdf", 
            format='pdf', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()
